# 02 — Differential Expression on Disease-Enriched Subclusters

Pseudobulk DESeq2 for each disease-enriched subcluster detected in
notebook 01, comparing subcluster cells vs the rest of the dataset.

**Workflow:**
1. Load the annotated AnnData from notebook 01
2. **Configure** DE parameters ← edit the configuration cell
3. For each subcluster: build pseudobulk, run DESeq2, generate volcano plots
4. Export per-subcluster DE tables for downstream analysis

**Multi-dataset usage:** Run this notebook once per dataset with the same
`DATASET_LABEL` used in notebook 01. DE results are saved to
`RESULTS_DIR / DATASET_LABEL / de/`.

## 1. Import Libraries

In [ ]:
from __future__ import annotations

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc

import scutils

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
sc.settings.verbosity = 1

## 2. Configuration

**Edit the cell below** before running the rest of the notebook.

Key parameters:
- `DATASET_LABEL` — must match the label used in notebook 01
- `PSEUDOBULK_GROUP_KEY` — column in `adata.obs` for biological replicates (e.g. `"sampleID"`)
- `RAW_COUNTS_LAYER` — layer with raw integer counts; `None` = use `adata.X`
- `LOG2FC_THRESH` / `PADJ_THRESH` — significance thresholds

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║                    CONFIGURATION — edit this cell                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Input / Output ────────────────────────────────────────────────────────────
DATASET_LABEL = "dataset_A"
RESULTS_DIR   = Path("results/disease_subclusters")

# Path to the AnnData saved by notebook 01
ADATA_PATH = RESULTS_DIR / DATASET_LABEL / "adata_with_subclusters.h5ad"

# ── Column names in adata.obs ─────────────────────────────────────────────────
RESULT_KEY           = "disease_subcluster"   # subcluster label column from notebook 01
PSEUDOBULK_GROUP_KEY = "sampleID"             # biological replicate column

# Layer that holds raw integer counts; set to None to use adata.X directly.
RAW_COUNTS_LAYER: str | None = "counts"

# ── DESeq2 settings ───────────────────────────────────────────────────────────
ALPHA      = 0.05    # DESeq2 significance threshold
SHRINK_LFC = True    # apply LFC shrinkage (recommended)

# ── Pseudobulk settings ───────────────────────────────────────────────────────
MIN_CELLS_PER_GROUP   = 10
MIN_SAMPLES_PER_GROUP = 2    # minimum replicates per group (test/ref)

# ── Significance thresholds for filtering / volcano plots ─────────────────────
LOG2FC_THRESH = 0.5
PADJ_THRESH   = 0.05
TOP_N_GENES   = 10    # genes to label on volcano plots

# ── Plotting ──────────────────────────────────────────────────────────────────
SAVE_FIGS = True

print("Configuration loaded.")
print(f"  Dataset label       : {DATASET_LABEL}")
print(f"  AnnData path        : {ADATA_PATH}")
print(f"  Pseudobulk group key: {PSEUDOBULK_GROUP_KEY}")
print(f"  Raw counts layer    : {RAW_COUNTS_LAYER!r}")
print(f"  log2FC threshold    : {LOG2FC_THRESH}")
print(f"  padj threshold      : {PADJ_THRESH}")

## 3. Load Data

In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
print(adata)

# List detected subclusters
labels = adata.obs[RESULT_KEY]
subclusters = sorted(labels[labels != "background"].unique().tolist())
print(f"\nDetected subclusters ({len(subclusters)}):")
for sc_label in subclusters:
    n = (labels == sc_label).sum()
    print(f"  • {sc_label}: {n:,} cells")

### Pre-flight: Replicate Availability

Check which subclusters have enough biological replicates for pseudobulk
DE before running the full pipeline.

In [ ]:
print(f"Checking replicate availability (MIN_CELLS_PER_GROUP={MIN_CELLS_PER_GROUP}, "
      f"MIN_SAMPLES_PER_GROUP={MIN_SAMPLES_PER_GROUP}):\n")

preflight_rows = []
for sc_label in subclusters:
    mask_test = adata.obs[RESULT_KEY] == sc_label
    mask_ref  = ~mask_test

    # Count samples with enough cells in each group
    test_samples = (
        adata.obs.loc[mask_test]
        .groupby(PSEUDOBULK_GROUP_KEY)
        .size()
        .pipe(lambda s: (s >= MIN_CELLS_PER_GROUP).sum())
    )
    ref_samples = (
        adata.obs.loc[mask_ref]
        .groupby(PSEUDOBULK_GROUP_KEY)
        .size()
        .pipe(lambda s: (s >= MIN_CELLS_PER_GROUP).sum())
    )

    ok = test_samples >= MIN_SAMPLES_PER_GROUP and ref_samples >= MIN_SAMPLES_PER_GROUP
    status = "✓" if ok else "⚠  SKIP"
    preflight_rows.append({
        "subcluster": sc_label,
        "n_cells_test": int(mask_test.sum()),
        "n_cells_ref": int(mask_ref.sum()),
        "samples_test": int(test_samples),
        "samples_ref": int(ref_samples),
        "status": status,
    })
    if not ok:
        print(f"  ⚠  {sc_label}: test={test_samples} samples, ref={ref_samples} samples — will be skipped")

preflight_df = pd.DataFrame(preflight_rows)
n_ok = (preflight_df["status"] == "✓").sum()
print(f"\n{n_ok}/{len(subclusters)} subclusters have sufficient replicates for DE.\n")
display(preflight_df.style.applymap(
    lambda v: "color: red; font-weight: bold" if v == "⚠  SKIP" else "",
    subset=["status"],
))

## 4. Pseudobulk DE per Subcluster

For each detected subcluster, we:
1. Tag cells as `"test"` (in subcluster) vs `"ref"` (everything else)
2. Build pseudobulk aggregates per sample × group
3. Run DESeq2 via `scutils.tl.deseq2`
4. Generate a volcano plot

In [ ]:
output_dir = RESULTS_DIR / DATASET_LABEL / "de"
output_dir.mkdir(parents=True, exist_ok=True)

de_results: dict[str, pd.DataFrame] = {}

for sc_label in subclusters:
    print(f"\n{'='*60}")
    print(f"Processing: {sc_label}")
    print(f"{'='*60}")

    # ── Tag cells ─────────────────────────────────────────────────────
    adata.obs["_de_group"] = "ref"
    adata.obs.loc[adata.obs[RESULT_KEY] == sc_label, "_de_group"] = "test"

    n_test = (adata.obs["_de_group"] == "test").sum()
    n_ref  = (adata.obs["_de_group"] == "ref").sum()
    print(f"  test: {n_test:,} cells | ref: {n_ref:,} cells")

    # ── Pseudobulk ────────────────────────────────────────────────────
    try:
        pb = scutils.pp.pseudobulk(
            adata,
            sample_col=PSEUDOBULK_GROUP_KEY,
            groups_col="_de_group",
            layer=RAW_COUNTS_LAYER,
            min_cells=MIN_CELLS_PER_GROUP,
            skip_count_check=False,
        )
    except Exception as exc:
        print(f"  ⚠  Pseudobulk failed: {exc}")
        continue

    # Check replicate counts
    group_counts = pb.obs["_de_group"].value_counts()
    if group_counts.get("test", 0) < MIN_SAMPLES_PER_GROUP or group_counts.get("ref", 0) < MIN_SAMPLES_PER_GROUP:
        print(
            f"  ⚠  Skipping: fewer than {MIN_SAMPLES_PER_GROUP} replicates per group "
            f"after min_cells filter. Sizes: {group_counts.to_dict()}"
        )
        continue

    print(f"  Pseudobulk: {pb.n_obs} observations ({group_counts.to_dict()})")

    # ── DESeq2 ────────────────────────────────────────────────────────
    try:
        results = scutils.tl.deseq2(
            pb,
            design="~_de_group",
            contrast=["_de_group", "test", "ref"],
            alpha=ALPHA,
            shrink_lfc=SHRINK_LFC,
            ref_level=["_de_group", "ref"],
            quiet=True,
        )
    except Exception as exc:
        print(f"  ✗ DESeq2 failed: {exc}")
        continue

    n_sig = int((results["padj"] < PADJ_THRESH).sum())
    n_up  = int(((results["padj"] < PADJ_THRESH) & (results["log2FoldChange"] > LOG2FC_THRESH)).sum())
    n_dn  = int(((results["padj"] < PADJ_THRESH) & (results["log2FoldChange"] < -LOG2FC_THRESH)).sum())
    print(f"  ✓ Significant genes: {n_sig} (up: {n_up}, down: {n_dn})")

    # Add gene column and subcluster metadata
    results = results.reset_index()
    results = results.rename(columns={results.columns[0]: "gene"})
    results["subcluster"] = sc_label
    de_results[sc_label] = results

    # ── Save DE table ─────────────────────────────────────────────────
    safe_label = sc_label.replace("|", "_").replace(" ", "_")
    csv_path = output_dir / f"de_{safe_label}.csv"
    results.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path}")

# Clean up temporary column
if "_de_group" in adata.obs.columns:
    del adata.obs["_de_group"]

print(f"\n✓ DE completed for {len(de_results)}/{len(subclusters)} subclusters.")

## 5. Volcano Plots

In [ ]:
for sc_label, results in de_results.items():
    # Format for volcano plot
    df_volcano = scutils.tl.format_deseq2_results(
        results.set_index("gene"),
    )

    fig = scutils.pl.volcano_plot(
        df_volcano,
        pval_cutoff=PADJ_THRESH,
        lfc_cutoff=LOG2FC_THRESH,
        top_n_up=TOP_N_GENES,
        top_n_down=TOP_N_GENES,
        figsize=(10, 7),
    )
    fig.axes[0].set_title(f"{sc_label}", fontsize=11, pad=10)
    fig.tight_layout()

    if SAVE_FIGS:
        safe_label = sc_label.replace("|", "_").replace(" ", "_")
        fig_path = output_dir / f"volcano_{safe_label}.png"
        fig.savefig(fig_path, bbox_inches="tight")
        print(f"Saved: {fig_path}")

    plt.show()
    plt.close(fig)

## 6. Summary of Top Upregulated Genes

In [ ]:
if de_results:
    combined = pd.concat(de_results.values(), ignore_index=True)

    upreg = (
        combined[
            (combined["padj"] < PADJ_THRESH)
            & (combined["log2FoldChange"] > LOG2FC_THRESH)
        ]
        .sort_values("padj")
        .groupby("subcluster")
        .head(10)
    )

    if not upreg.empty:
        print(f"Top 10 upregulated genes per subcluster ({len(upreg)} rows):")
        display(
            upreg[["subcluster", "gene", "log2FoldChange", "padj"]]
            .style
            .format({"log2FoldChange": "{:.3f}", "padj": "{:.2e}"})
            .background_gradient(subset="log2FoldChange", cmap="Reds")
        )
    else:
        print("No significantly upregulated genes found.")

    # Save combined table
    combined_path = output_dir / "de_all_subclusters.csv"
    combined.to_csv(combined_path, index=False)
    print(f"\nSaved combined table: {combined_path}")
else:
    print("No DE results to display.")

---

## Summary

| Step | Function | Key parameters |
|------|----------|----------------|
| Pseudobulk | `scutils.pp.pseudobulk()` | `PSEUDOBULK_GROUP_KEY`, `MIN_CELLS_PER_GROUP` |
| DE analysis | `scutils.tl.deseq2()` | `ALPHA`, `SHRINK_LFC` |
| Volcano plot | `scutils.pl.volcano_plot()` | `LOG2FC_THRESH`, `PADJ_THRESH`, `TOP_N_GENES` |
| Export | `pd.DataFrame.to_csv()` | `RESULTS_DIR / DATASET_LABEL / de/` |

**Next:** Run `03_functional_enrichment.ipynb` to annotate subclusters
based on pathway enrichment of DE genes.